# Análise de escalonamento das variáveis quantitativas

Notebook de apoio ao Anexo I da documentação. Dois materiais são produzidos:

- as medidas de referência usadas no escalonamento, apresentadas na Subseção A.2.2, com desvio padrão populacional;
- os histogramas da temperatura antes e depois da normalização Min-Max, por área térmica, apresentados na Subseção A.2.4.

A base utilizada é a mesma exportada pelo notebook de leitura e limpeza, de modo que todas as análises do projeto partam exatamente do mesmo conjunto de leituras válidas, já com a correção das células gravadas como porcentagem e com o descarte do termopar Oven RTD Connector Body.

In [ ]:
# == caminhos do projeto ==
# A pasta deste caderno, dentro de notebooks/. Os CSVs sao lidos e gravados
# nela; as leituras entre pastas usam ../<outra pasta>. Para que esses caminhos
# valham qualquer que seja o diretorio em que o kernel foi aberto, o bloco
# abaixo localiza a raiz do repositorio -- a pasta que contem notebooks/ e
# documents/ -- e entra na pasta do caderno. No Colab, onde nao existe
# repositorio e os arquivos ficam soltos ao lado do caderno, nada e alterado.
import os

PASTA_CADERNO = "colabs_S1_S2"


def raiz_repositorio(inicio=None):
    """Sobe a arvore de diretorios ate a raiz do repositorio; None se nao achar."""
    atual = os.path.abspath(inicio or os.getcwd())
    while True:
        if all(os.path.isdir(os.path.join(atual, p)) for p in ("notebooks", "documents")):
            return atual
        pai = os.path.dirname(atual)
        if pai == atual:
            return None
        atual = pai


RAIZ_REPO = raiz_repositorio()
if RAIZ_REPO:
    os.chdir(os.path.join(RAIZ_REPO, "notebooks", PASTA_CADERNO))
print("Diretorio de trabalho:", os.path.abspath("."))

In [ ]:
# Importação de bibliotecas
import os
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

## Preparação da base

A base limpa é carregada a partir da exportação do notebook de leitura, e não reconstruída a partir da planilha bruta. Dessa forma, as correções e os descartes aplicados na limpeza valem também para esta análise, evitando que o Anexo I e a Seção 4.2 apresentem números divergentes para o mesmo conjunto de dados.

In [ ]:
DIR_SAIDA = "dados_intermediarios"
CAMINHO_LIMPO = os.path.join(DIR_SAIDA, "df_limpo.csv")

if not os.path.exists(CAMINHO_LIMPO):
    raise FileNotFoundError(
        "df_limpo.csv não encontrado. Execute o notebook leitura.ipynb antes deste."
    )

df_limpo = pd.read_csv(CAMINHO_LIMPO)
df_limpo["Possui Active Limit"] = df_limpo["Possui Active Limit"].astype(bool)

print("Base carregada:", df_limpo.shape)
print("Termopares:", df_limpo["TC Name"].nunique())

In [ ]:

REGIOES = [
    "Mainback",
    "Coil Temps",
    "Main Vent Sys",
    "IPC Cooling",
    "IPC Comp",
    "Oven HMI",
    "CT Knobs",
]

ROTULOS_REGIAO = {
    "Mainback": "Mainback - peças da parede traseira",
    "Coil Temps": "Coil Temps - bobinas do cooktop",
    "Main Vent Sys": "Main Vent Sys - ventilação principal",
    "IPC Cooling": "IPC Cooling - ventilação cooktop/IPC",
    "IPC Comp": "IPC Comp - componentes da placa IPC",
    "Oven HMI": "Oven HMI - interface do forno",
    "CT Knobs": "CT Knobs - encoders do cooktop",
}

BINS = "sturges"

faltantes = [r for r in df_limpo["Key Area"].unique() if r not in REGIOES]
assert not faltantes, f"Áreas presentes na base e ausentes da lista: {faltantes}"

print(f"Áreas térmicas: {len(REGIOES)}")

## A.2.2 Medidas de referência para o escalonamento

O valor mínimo e o valor máximo servem de base para a normalização Min-Max; a média e o desvio padrão fundamentam a padronização por escore-z. O desvio padrão reportado é o populacional.

In [ ]:
def medidas(serie):
    return {
        "Count": len(serie),
        "Média": round(serie.mean(), 2),
        "Desvio Padrão (populacional)": round(serie.std(ddof=0), 2),
        "Mínimo": round(serie.min(), 2),
        "Máximo": round(serie.max(), 2),
    }


serie_completa = df_limpo["Temperature"]

tabela_variavel = pd.DataFrame([{"Variável": "Temperatura", **medidas(serie_completa)}])
tabela_regioes = pd.DataFrame(
    [
        {"Região": regiao, **medidas(df_limpo.loc[df_limpo["Key Area"] == regiao, "Temperature"])}
        for regiao in REGIOES
    ]
)

assert tabela_regioes["Count"].sum() == len(serie_completa)

print(tabela_variavel.to_string(index=False))
print()
print(tabela_regioes.to_string(index=False))

## A.2.4 Histogramas antes e depois da normalização Min-Max

Cada figura reúne dois painéis da mesma amostra: à esquerda a escala original em graus Celsius, à direita a escala adimensional restrita ao intervalo de 0 a 1. Como a normalização é uma transformação linear, as alturas das barras são idênticas nos dois painéis; o que muda é a unidade do eixo horizontal.

In [ ]:
from pathlib import Path

DESTINO = Path("../../documents/assets/histogramas/posEscalonamento")
DESTINO.mkdir(parents=True, exist_ok=True)

ARQUIVOS = {
    "Mainback": "mainback.png",
    "Coil Temps": "coilTemps.png",
    "Main Vent Sys": "mainVentSys.png",
    "IPC Cooling": "ipcCooling.png",
    "IPC Comp": "ipcComp.png",
    "Oven HMI": "ovenHmi.png",
    "CT Knobs": "ctKnobs.png",
}

for regiao in REGIOES:
    serie = df_limpo.loc[df_limpo["Key Area"] == regiao, "Temperature"]
    minimo, maximo = serie.min(), serie.max()
    normalizada = (serie - minimo) / (maximo - minimo)

    figura, (esquerda, direita) = plt.subplots(1, 2, figsize=(14.19, 4.9))
    paineis = (
        (esquerda, serie, "#4C72B0", r"$\bf{ANTES}$ - escala original",
         "Temperatura (°C)", 1, " °C"),
        (direita, normalizada, "#DD8452", r"$\bf{DEPOIS}$ - Min-Max [0, 1]",
         "Temperatura normalizada (adimensional)", 3, ""),
    )

    for eixo, dados, cor, titulo, rotulo_x, casas, sufixo in paineis:
        eixo.hist(dados, bins=BINS, edgecolor="black", color=cor)
        eixo.axvline(dados.mean(), color="#55A868", linewidth=1.8,
                     label=f"Média = {dados.mean():.{casas}f}{sufixo}")
        eixo.axvline(dados.median(), color="#8172B2", linestyle=":", linewidth=1.8,
                     label=f"Mediana = {dados.median():.{casas}f}{sufixo}")
        eixo.set_title(titulo, fontsize=11)
        eixo.set_xlabel(rotulo_x)
        eixo.set_ylabel("Frequência (nº de leituras)")
        eixo.legend(fontsize=8)

    figura.suptitle(f"{ROTULOS_REGIAO[regiao]} (n = {len(serie)})", fontsize=13)
    plt.tight_layout()
    figura.savefig(DESTINO / ARQUIVOS[regiao], dpi=100)
    plt.show()